In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Generic Parameterized Transformation
# MAGIC Cleanses Bronze data and applies Delta Lake ACID Merges into `silver.cln_<entity>`.


# COMMAND ----------
# 2. Declare parameters via Databricks Widgets
dbutils.widgets.text("p_entity_name", "plans", "Entity Name")
dbutils.widgets.text("p_run_mode", "incremental", "Run Mode (full/incremental)")

entity_name = dbutils.widgets.get("p_entity_name").strip().lower()
run_mode = dbutils.widgets.get("p_run_mode").strip().lower()

print(f"Executing Silver transformation for entity: '{entity_name}' in mode: '{run_mode}'")

# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# COMMAND ----------
# 3. Dynamic path resolution based on entity metadata
CATALOG = "adb_saas_dev"
SILVER_BASE = "abfss://silver@datalakesaas.dfs.core.windows.net"

entity_paths = {
    "plans": {"bronze_table": f"{CATALOG}.bronze.raw_plans", "table": f"{CATALOG}.silver.cln_plans", "location": f"{SILVER_BASE}/cln_plans", "pk": ["plan_id"]},
    "organizations": {"bronze_table": f"{CATALOG}.bronze.raw_organizations", "table": f"{CATALOG}.silver.cln_organizations", "location": f"{SILVER_BASE}/cln_organizations", "pk": ["org_id"]},
    "users": {"bronze_table": f"{CATALOG}.bronze.raw_users", "table": f"{CATALOG}.silver.cln_users", "location": f"{SILVER_BASE}/cln_users", "pk": ["user_id"]},
    "subscriptions": {"bronze_table": f"{CATALOG}.bronze.raw_subscriptions", "table": f"{CATALOG}.silver.cln_subscriptions", "location": f"{SILVER_BASE}/cln_subscriptions", "pk": ["subscription_id"]},
    "invoices": {"bronze_table": f"{CATALOG}.bronze.raw_invoices", "table": f"{CATALOG}.silver.cln_invoices", "location": f"{SILVER_BASE}/cln_invoices", "pk": ["invoice_id"]},
    "feature_usage": {"bronze_table": f"{CATALOG}.bronze.raw_feature_usage", "table": f"{CATALOG}.silver.cln_feature_usage", "location": f"{SILVER_BASE}/cln_feature_usage", "pk": ["user_id", "org_id", "usage_date", "feature_name"]},
    "logins": {"bronze_table": f"{CATALOG}.bronze.raw_logins", "table": f"{CATALOG}.silver.cln_logins", "location": f"{SILVER_BASE}/cln_logins", "pk": ["event_id"]}
}

if entity_name not in entity_paths:
    raise ValueError(f"Entity '{entity_name}' is not recognized in silver transformation mapping.")

cfg = entity_paths[entity_name]

# Read raw Bronze data from Unity Catalog external tables
# (logins entity reads from its UC table separately in the logins branch)
if entity_name != "logins":
    df_raw = spark.table(cfg["bronze_table"])
    print(f"Bronze records read: {df_raw.count()}")

# COMMAND ----------
# 4. Entity-specific transformations (replicates clean_sources.py logic)

if entity_name == "plans":
    df_clean = df_raw \
        .withColumn("plan_id", F.col("plan_id").cast("int")) \
        .withColumn("plan_name", F.trim(F.col("plan_name"))) \
        .withColumn("tier", F.lower(F.trim(F.col("tier")))) \
        .withColumn("monthly_fee", F.coalesce(F.col("monthly_fee").cast("decimal(10,2)"), F.lit(0.00))) \
        .withColumn("created_at", F.to_timestamp(F.col("created_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("plan_id", "plan_name", "tier", "monthly_fee", "created_at", "updated_at")

elif entity_name == "organizations":
    # ISO-2 Country Standardizer Map
    country_map = F.create_map([
        F.lit("USA"), F.lit("US"), F.lit("United States"), F.lit("US"), F.lit("US"), F.lit("US"),
        F.lit("CAN"), F.lit("CA"), F.lit("Canada"), F.lit("CA"), F.lit("CA"), F.lit("CA"),
        F.lit("GBR"), F.lit("GB"), F.lit("United Kingdom"), F.lit("GB"), F.lit("GB"), F.lit("GB"),
        F.lit("DEU"), F.lit("DE"), F.lit("Germany"), F.lit("DE"), F.lit("DE"), F.lit("DE"),
        F.lit("IND"), F.lit("IN"), F.lit("India"), F.lit("IN"), F.lit("IN"), F.lit("IN")
    ])

    df_clean = df_raw \
        .withColumn("org_id", F.col("org_id").cast("int")) \
        .withColumn("name", F.trim(F.col("name"))) \
        .withColumn("industry", F.coalesce(F.when(F.trim(F.col("industry")) == "", "Unknown").otherwise(F.trim(F.col("industry"))), F.lit("Unknown"))) \
        .withColumn("country_code", F.coalesce(country_map[F.col("country")], F.lit("XX"))) \
        .withColumn("is_active", F.col("is_active").cast("boolean")) \
        .withColumn("created_at", F.to_timestamp(F.col("created_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("org_id", "name", "industry", "country_code", "is_active", "created_at", "updated_at")

elif entity_name == "users":
    # Email syntax validation pattern
    email_regex = r"^[\w\.-]+@[\w\.-]+\.\w+$"

    df_clean = df_raw \
        .withColumn("user_id", F.col("user_id").cast("int")) \
        .withColumn("org_id", F.coalesce(F.col("org_id").cast("int"), F.lit(-1))) \
        .withColumn("cleaned_email", F.lower(F.trim(F.col("email")))) \
        .withColumn("is_valid_email", F.col("cleaned_email").rlike(email_regex)) \
        .withColumn("email", F.when(F.col("is_valid_email"), F.col("cleaned_email")).otherwise("invalid_user@system.local")) \
        .withColumn("role", F.coalesce(F.trim(F.col("role")), F.lit("Member"))) \
        .withColumn("is_active", F.col("is_active").cast("boolean")) \
        .withColumn("created_at", F.to_timestamp(F.col("created_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("user_id", "org_id", "email", "is_valid_email", "role", "is_active", "created_at", "updated_at")

elif entity_name == "subscriptions":
    df_clean = df_raw \
        .withColumn("subscription_id", F.col("subscription_id").cast("int")) \
        .withColumn("org_id", F.col("org_id").cast("int")) \
        .withColumn("plan_id", F.col("plan_id").cast("int")) \
        .withColumn("status", F.lower(F.trim(F.col("status")))) \
        .withColumn("started_at", F.to_timestamp(F.col("started_at"))) \
        .withColumn("cancelled_at", F.to_timestamp(F.col("cancelled_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("subscription_id", "org_id", "plan_id", "status", "started_at", "cancelled_at", "updated_at")

elif entity_name == "invoices":
    df_clean = df_raw \
        .withColumn("invoice_id", F.col("invoice_id").cast("int")) \
        .withColumn("subscription_id", F.col("subscription_id").cast("int")) \
        .withColumn("amount_due", F.coalesce(F.col("amount_due").cast("decimal(10,2)"), F.lit(0.00))) \
        .withColumn("amount_paid", F.coalesce(F.col("amount_paid").cast("decimal(10,2)"), F.lit(0.00))) \
        .withColumn("status", F.lower(F.trim(F.col("status")))) \
        .withColumn("billing_date", F.to_date(F.col("billing_date"))) \
        .withColumn("paid_at", F.to_timestamp(F.col("paid_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .withColumn("is_anomaly", F.col("amount_paid") > F.col("amount_due")) \
        .select("invoice_id", "subscription_id", "amount_due", "amount_paid", "status", "billing_date", "paid_at", "is_anomaly", "updated_at")

elif entity_name == "feature_usage":
    # Parse mixed date formats without crashing (YYYY-MM-DD vs MM/DD/YYYY vs DD-MM-YYYY)
    raw_date_col = F.col("UsageDate")
    parsed_date = F.coalesce(
        F.expr("try_to_date(UsageDate, 'yyyy-MM-dd')"),
        F.expr("try_to_date(UsageDate, 'MM/dd/yyyy')"),
        F.expr("try_to_date(UsageDate, 'dd-MM-yyyy')")
    )

    # Strip currency signs, commas, and negative values
    cleaned_usage_count = F.regexp_replace(F.col("UsageCount").cast("string"), r"[\$,]", "")

    df_clean = df_raw \
        .withColumnRenamed("UserId", "user_id") \
        .withColumnRenamed("OrgId", "org_id") \
        .withColumnRenamed("FeatureName", "feature_name") \
        .withColumn("user_id", F.coalesce(F.col("user_id").cast("int"), F.lit(-1))) \
        .withColumn("org_id", F.coalesce(F.col("org_id").cast("int"), F.lit(-1))) \
        .withColumn("usage_date", parsed_date) \
        .withColumn("feature_name", F.coalesce(F.when(F.trim(F.col("feature_name")) == "", "unknown_feature").otherwise(F.trim(F.col("feature_name"))), F.lit("unknown_feature"))) \
        .withColumn("usage_count", F.coalesce(cleaned_usage_count.cast("int"), F.lit(0))) \
        .filter(F.col("usage_count") >= 0) \
        .dropDuplicates(["user_id", "org_id", "usage_date", "feature_name"]) \
        .select("user_id", "org_id", "usage_date", "feature_name", "usage_count")

elif entity_name == "logins":
    # 1. Read the raw JSON data from the Unity Catalog bronze table
    df_raw_json = spark.table(cfg["bronze_table"])
    
    # 2. Explode the nested 'data' array so every event becomes a distinct row
    df_exploded = df_raw_json.select(
        F.explode("data").alias("login_event"),
        F.col("_ingested_at").alias("_batch_ingested_at") if "_ingested_at" in df_raw_json.columns else F.current_timestamp().alias("_batch_ingested_at")
    ).select(
        F.col("login_event.event_id").alias("event_id"),
        F.col("login_event.user_uid").alias("user_uid"),
        F.col("login_event.ip_address").alias("ip_address"),
        F.col("login_event.client_device").alias("client_device"),
        F.col("login_event.login_success").alias("login_success"),
        F.col("login_event.event_timestamp").alias("event_timestamp")
    )
    
    print(f"Raw exploded login events: {df_exploded.count()}")  # Will display 10,300!
    
    # 3. Apply the Data Quality & Cleaning Rules (Deduplication, UTC parsing, Null handling)
    # Deduplicate network retries retaining newest event_timestamp
    window_spec = Window.partitionBy("event_id").orderBy(F.col("event_timestamp").desc())
    
    df_clean = df_exploded \
        .withColumnRenamed("user_uid", "user_id") \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num") \
        .withColumn("user_id", F.coalesce(F.col("user_id").cast("int"), F.lit(-1))) \
        .withColumn("ip_address", F.trim(F.col("ip_address"))) \
        .withColumn("client_device", F.coalesce(F.trim(F.col("client_device")), F.lit("Unknown"))) \
        .withColumn("login_success", F.col("login_success").cast("boolean")) \
        .withColumn("event_timestamp", F.to_utc_timestamp(F.to_timestamp(F.col("event_timestamp")), "UTC")) \
        .filter(F.col("event_timestamp").isNotNull()) \
        .select("event_id", "user_id", "ip_address", "client_device", "login_success", "event_timestamp")

# COMMAND ----------
# 5. Unity Catalog Delta Table Upsert

target_table = cfg["table"]
target_location = cfg["location"]
pk_cols = cfg["pk"]
df_clean = df_clean.dropDuplicates(pk_cols)
match_predicate = " AND ".join([f"target.{col} = source.{col}" for col in pk_cols])

if not spark.catalog.tableExists(target_table):
    print(f"Creating external table {target_table} at {target_location}...")
    df_clean.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", target_location) \
        .saveAsTable(target_table)
    print(f"Table {target_table} created.")
else:
    print(f"Merging updates into {target_table}...")
    delta_target = DeltaTable.forName(spark, target_table)
    
    delta_target.alias("target").merge(
        df_clean.alias("source"),
        match_predicate
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Merge completed successfully.")

# COMMAND ----------
# 6. Audit & Time Travel Verification
display(spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 5"))